
# DIA Web Scraper

This notebook collects product and price information from DIA using Selenium.

### How it works

1. Searches for the 20 product terms defined in `SEARCH_TERMS`.
2. Loads product cards using internal scrolling and automatic retries.
3. Extracts product names, IDs, prices, units, URLs, and timestamps.
4. Combines the search results into a RAW dataset.
5. Validates collection completeness and saves RAW files.

### Data Storage

- Current RAW: `data/raw/dia_products_raw.csv`
- Historical RAW: `data/history/dia/YYYY-MM-DD_raw.csv`

The central pipeline handles deduplication, standardization, classification, and historical price processing.

### Collection Policy

- Complete collections update the current RAW and create a historical snapshot.
- Partial collections are archived without replacing the current RAW.
- Existing historical snapshots are never overwritten.
- TEST_MODE runs selected searches without saving files.


In [ ]:
import time
import re
from pathlib import Path
from datetime import datetime

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

HOME_URL = "https://www.dia.es/"
CARD_SELECTOR = 'div[data-test-id="product-card"]'
SCROLL_SELECTOR = ".search-component__content"

WAIT = 20
MAX_BATCHES = 40
NEXT_BATCH_TIMEOUT = 15
MAX_RETRIES = 2

SEARCH_TERMS = [
    "leche", "huevos", "manzana", "plátano", "tomate", "arroz",
    "pasta", "pan", "pollo", "carne picada", "salmón", "yogur",
    "queso", "agua", "zumo", "atún", "garbanzos", "aceite",
    "detergente", "papel higiénico"
]

# --------------------------------------------------
# TEST MODE
# --------------------------------------------------

TEST_MODE = False
TEST_TERMS = ["leche"]

TERMS_TO_SCRAPE = TEST_TERMS if TEST_MODE else SEARCH_TERMS

print("Search terms:", TERMS_TO_SCRAPE)

Search terms: ['leche']


In [28]:

# --------------------------------------------------
# PROJECT DIRECTORIES
# --------------------------------------------------

def find_project_root():
    current = Path.cwd().resolve()

    for path in (current, *current.parents):
        if path.name == "supermarket_project":
            return path

    if current.name == "scrapers":
        return current.parent

    raise FileNotFoundError("Could not locate the project root.")


BASE_DIR = find_project_root()

RAW_DIR = BASE_DIR / "data" / "raw"
HISTORY_DIR = BASE_DIR / "data" / "history" / "dia"

RAW_PATH = RAW_DIR / "dia_products_raw.csv"

RUN_DATE = datetime.now().strftime("%Y-%m-%d")
RAW_HISTORY_PATH = HISTORY_DIR / f"{RUN_DATE}_raw.csv"

print("Project:", BASE_DIR)
print("Collection date:", RUN_DATE)
print("Current RAW:", RAW_PATH)
print("Historical RAW:", RAW_HISTORY_PATH)

Project: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project
Collection date: 2026-11-13
Current RAW: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\raw\dia_products_raw.csv
Historical RAW: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\history\dia\2026-11-13_raw.csv


In [29]:
def start_driver():
    options = Options()
    options.add_argument("--window-size=1920,1080")
    driver = webdriver.Chrome(options=options)
    driver.get(HOME_URL)
    return driver


def accept_cookies_if_needed(driver):
    try:
        button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable(
                (By.XPATH, "//button[contains(., 'Aceptar')]")
            )
        )
        button.click()
        print("Cookies aceptadas.")
    except TimeoutException:
        pass


def go_to_home(driver):
    driver.get(HOME_URL)
    WebDriverWait(driver, WAIT).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )
    time.sleep(1)
    accept_cookies_if_needed(driver)


def assert_no_security_challenge(driver):
    text = driver.find_element(By.TAG_NAME, "body").text.lower()
    messages = [
        "confirma que no eres un robot",
        "comprueba que eres humano",
        "access denied",
        "captcha",
    ]
    if any(message in text for message in messages):
        raise RuntimeError("Verificación de seguridad detectada.")

In [30]:
def find_search_box(driver):
    selectors = [
        'input[type="search"]',
        'input[data-test-id*="search"]',
        'input[data-testid*="search"]',
        'input[placeholder*="Buscar"]',
        'input[placeholder*="buscar"]',
        'input[aria-label*="Buscar"]',
        'input[aria-label*="buscar"]',
    ]

    def visible_candidate(_):
        for selector in selectors:
            for element in driver.find_elements(By.CSS_SELECTOR, selector):
                if element.is_displayed() and element.is_enabled():
                    return element
        return False

    return WebDriverWait(driver, WAIT).until(visible_candidate)


def search_dia(driver, term):
    box = find_search_box(driver)
    box.click()
    box.send_keys(Keys.CONTROL, "a")
    box.send_keys(Keys.BACKSPACE)
    box.send_keys(term)
    box.send_keys(Keys.ENTER)

    WebDriverWait(driver, WAIT).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, SCROLL_SELECTOR))
    )

    WebDriverWait(driver, WAIT).until(
        lambda d: len(d.find_elements(By.CSS_SELECTOR, CARD_SELECTOR)) > 0
    )

    assert_no_security_challenge(driver)
    time.sleep(1)

In [31]:
def load_all_cards(driver):
    container = driver.find_element(By.CSS_SELECTOR, SCROLL_SELECTOR)
    count = len(driver.find_elements(By.CSS_SELECTOR, CARD_SELECTOR))
    print("Tarjetas iniciales:", count)

    for batch in range(1, MAX_BATCHES + 1):
        previous = count

        state = driver.execute_script(
            '''
            const el = arguments[0];
            return {
                scrollHeight: el.scrollHeight,
                clientHeight: el.clientHeight
            };
            ''',
            container
        )

        target = max(
            0,
            state["scrollHeight"] - state["clientHeight"] - 450
        )

        driver.execute_script(
            "arguments[0].scrollTop = arguments[1];",
            container,
            target
        )

        try:
            WebDriverWait(
                driver,
                NEXT_BATCH_TIMEOUT,
                poll_frequency=0.3
            ).until(
                lambda d: len(
                    d.find_elements(By.CSS_SELECTOR, CARD_SELECTOR)
                ) > previous
            )

            count = len(
                driver.find_elements(By.CSS_SELECTOR, CARD_SELECTOR)
            )

            print(f"Tanda {batch}: {previous} -> {count}")

        except TimeoutException:
            return previous, "complete"

    return count, "max_batches_reached"

In [32]:
def clean_price(text):
    if not text:
        return None

    match = re.search(
        r"\d+(?:[.,]\d+)?",
        str(text).replace("\xa0", " ")
    )

    return (
        float(match.group().replace(",", "."))
        if match
        else None
    )


def extract_price_per_unit(text):
    if not text:
        return None, None

    value = clean_price(text)

    match = re.search(
        r"€\s*/\s*([A-Za-zÁÉÍÓÚáéíóú]+)",
        text
    )

    unit = match.group(1).upper() if match else None
    return value, unit


def extract_products(driver, searched_item):
    soup = BeautifulSoup(driver.page_source, "html.parser")
    cards = soup.select(CARD_SELECTOR)

    rows = []
    scraped_at = datetime.now().isoformat(timespec="seconds")

    for card in cards:
        link = card.select_one(
            '[data-test-id="search-product-card-name"]'
        )
        price_el = card.select_one(
            '[data-test-id="search-product-card-unit-price"]'
        )
        ppu_el = card.select_one(
            '[data-test-id="search-product-card-kilo-price"]'
        )

        if not link or not price_el:
            continue

        price_raw = price_el.get_text(" ", strip=True)
        ppu_raw = (
            ppu_el.get_text(" ", strip=True)
            if ppu_el
            else None
        )

        ppu, unit = extract_price_per_unit(ppu_raw)

        rows.append({
            "searched_item": searched_item,
            "supermarket": "DIA",
            "product_id": card.get("object_id"),
            "name": link.get_text(" ", strip=True),
            "price_raw": price_raw,
            "price": clean_price(price_raw),
            "price_per_unit_raw": ppu_raw,
            "price_per_unit": ppu,
            "unit": unit,
            "url": link.get("href"),
            "scraped_at": scraped_at,
        })

    return pd.DataFrame(rows)

In [33]:
driver = start_driver()
time.sleep(2)
accept_cookies_if_needed(driver)

all_results = []
summary = []

for i, term in enumerate(TERMS_TO_SCRAPE, start=1):
    success = False
    started = time.monotonic()

    for attempt in range(1, MAX_RETRIES + 2):
        print("\n" + "=" * 70)
        print(f"[{i}/{len(TERMS_TO_SCRAPE)}] {term} · attempt {attempt}")

        try:
            go_to_home(driver)
            search_dia(driver, term)

            loaded, load_status = load_all_cards(driver)
            df_term = extract_products(driver, term)
            extracted = len(df_term)

            if load_status != "complete":
                raise RuntimeError(
                    f"La carga terminó con estado '{load_status}'."
                )

            if extracted == 0 or extracted != loaded:
                raise RuntimeError(
                    f"Tarjetas cargadas {loaded}, filas extraídas {extracted}."
                )

            all_results.append(df_term)

            summary.append({
                "searched_item": term,
                "cards_loaded": loaded,
                "rows_extracted": extracted,
                "attempts": attempt,
                "status": "OK",
                "seconds": round(time.monotonic() - started, 2),
            })

            print("Estado: OK")
            success = True
            break

        except Exception as exc:
            print(f"Intento {attempt} fallido: {type(exc).__name__}: {exc}")
            if attempt < MAX_RETRIES + 1:
                time.sleep(2)

    if not success:
        summary.append({
            "searched_item": term,
            "cards_loaded": None,
            "rows_extracted": 0,
            "attempts": MAX_RETRIES + 1,
            "status": "ERROR",
            "seconds": round(time.monotonic() - started, 2),
        })

df_summary = pd.DataFrame(summary)
display(df_summary)

Cookies aceptadas.

[1/1] leche · attempt 1
Tarjetas iniciales: 35
Tanda 1: 35 -> 65
Tanda 2: 65 -> 95
Tanda 3: 95 -> 125
Tanda 4: 125 -> 155
Tanda 5: 155 -> 185
Tanda 6: 185 -> 215
Tanda 7: 215 -> 245
Tanda 8: 245 -> 275
Tanda 9: 275 -> 305
Tanda 10: 305 -> 335
Tanda 11: 335 -> 365
Tanda 12: 365 -> 395
Tanda 13: 395 -> 422
Estado: OK


,searched_item,cards_loaded,rows_extracted,attempts,status,seconds
0,leche,422,422,1,OK,61.45


## RAW — Combine All Search Results into a Single Dataset

In [34]:
# --------------------------------------------------
# PREPARE RAW DATA
# --------------------------------------------------

if not all_results:
    raise RuntimeError("No products were collected.")

df_dia_raw = pd.concat(
    all_results,
    ignore_index=True
)

print("Raw rows:", len(df_dia_raw))
print("Unique product IDs:", df_dia_raw["product_id"].nunique())
print("Raw data ready for processing.")


Raw rows: 422
Unique product IDs: 417
Raw data ready for processing.


In [35]:
driver.quit()
print("Driver cerrado.")

Driver cerrado.


In [36]:
# --------------------------------------------------
# VALIDATE AND SAVE CURRENT AND HISTORICAL RAW DATA
# --------------------------------------------------

import os
import tempfile

if TEST_MODE:
    print("TEST MODE: no files will be saved.")

else:
    # --------------------------------------------------
    # VALIDATE COLLECTED DATA
    # --------------------------------------------------

    if df_dia_raw.empty:
        raise RuntimeError(
            "No products collected. No files saved."
        )

    if df_summary.empty:
        raise RuntimeError(
            "No search results were recorded. No files saved."
        )

    required_columns = [
        "searched_item",
        "supermarket",
        "product_id",
        "name",
        "quantity_raw",
        "price_raw",
        "price",
        "price_per_unit_raw",
        "price_per_unit",
        "unit",
        "url",
        "scraped_at",
    ]

    missing_columns = [
        column for column in required_columns
        if column not in df_dia_raw.columns
    ]

    if missing_columns:
        raise RuntimeError(
            f"Missing RAW columns: {missing_columns}"
        )

    # --------------------------------------------------
    # CHECK COLLECTION COMPLETENESS
    # --------------------------------------------------

    all_terms_attempted = (
        len(df_summary) == len(SEARCH_TERMS)
        and set(df_summary["searched_item"]) == set(SEARCH_TERMS)
    )

    successful_searches = df_summary["status"].eq("OK").all()

    collection_complete = bool(
        all_terms_attempted and successful_searches
    )

    # --------------------------------------------------
    # VALIDATE RAW PRODUCT DATA
    # --------------------------------------------------

    numeric_prices = pd.to_numeric(
        df_dia_raw["price"],
        errors="coerce"
    )

    valid_prices = (
        numeric_prices.notna()
        & numeric_prices.gt(0)
    )

    valid_names = (
        df_dia_raw["name"].notna()
        & df_dia_raw["name"].astype(str).str.strip().ne("")
    )

    if not valid_prices.any() or not valid_names.any():
        raise RuntimeError(
            "No valid products or selling prices found. "
            "No files saved."
        )

    invalid_rows = int(
        (~(valid_prices & valid_names)).sum()
    )

    if invalid_rows:
        print(
            f"WARNING: {invalid_rows} RAW rows contain "
            "missing names or invalid prices."
        )
        print(
            "These rows will be preserved for central "
            "pipeline validation."
        )

    # --------------------------------------------------
    # PROTECT EXISTING HISTORICAL SNAPSHOTS
    # --------------------------------------------------

    if RAW_HISTORY_PATH.exists():
        raise FileExistsError(
            "A RAW historical snapshot already exists "
            f"for {RUN_DATE}: {RAW_HISTORY_PATH}. "
            "No files were modified."
        )

    # --------------------------------------------------
    # PREPARE OUTPUT DIRECTORIES
    # --------------------------------------------------

    RAW_DIR.mkdir(parents=True, exist_ok=True)
    HISTORY_DIR.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------
    # PREPARE TEMPORARY RAW CSV
    # --------------------------------------------------

    def prepare_raw_temp(destination):
        fd, temp_name = tempfile.mkstemp(
            prefix=".dia_",
            suffix=".tmp",
            dir=destination.parent
        )

        try:
            with os.fdopen(
                fd,
                "w",
                encoding="utf-8-sig",
                newline=""
            ) as file:
                df_dia_raw.to_csv(
                    file,
                    index=False
                )

            return Path(temp_name)

        except Exception:
            Path(temp_name).unlink(missing_ok=True)
            raise

    history_temp = None
    current_temp = None

    try:
        history_temp = prepare_raw_temp(RAW_HISTORY_PATH)

        if collection_complete:
            current_temp = prepare_raw_temp(RAW_PATH)

        # Save the historical snapshot without overwriting it.
        with history_temp.open("rb") as source:
            with RAW_HISTORY_PATH.open("xb") as target:
                while True:
                    chunk = source.read(1024 * 1024)
                    if not chunk:
                        break
                    target.write(chunk)

        print("Saved historical RAW:", RAW_HISTORY_PATH)

        # Only complete collections update the current RAW.
        if collection_complete:
            os.replace(current_temp, RAW_PATH)
            print("Saved current RAW:", RAW_PATH)

    finally:
        if history_temp is not None:
            history_temp.unlink(missing_ok=True)

        if current_temp is not None:
            current_temp.unlink(missing_ok=True)

    # --------------------------------------------------
    # COLLECTION SUMMARY
    # --------------------------------------------------

    print("\nCOLLECTION SUMMARY")
    print("Collection date:", RUN_DATE)
    print("RAW rows:", len(df_dia_raw))
    print(
        "Distinct product IDs:",
        df_dia_raw["product_id"].nunique()
    )
    print("Searches attempted:", len(df_summary))
    print("Expected searches:", len(SEARCH_TERMS))
    print("Rows requiring validation:", invalid_rows)

    if collection_complete:
        print("\nStatus: COMPLETE")
        print("Current RAW and historical RAW saved.")

    else:
        print("\nStatus: PARTIAL")
        print(
            "Historical RAW saved. "
            "Current RAW was not modified."
        )

        failed_searches = df_summary[
            df_summary["status"] != "OK"
        ]

        if not failed_searches.empty:
            display(
                failed_searches[
                    [
                        "searched_item",
                        "status",
                        "attempts",
                        "rows_extracted",
                    ]
                ]
            )

        if not all_terms_attempted:
            print(
                "WARNING: Not all expected search "
                "terms were attempted."
            )

    print("\nSaved outputs:")
    print(RAW_HISTORY_PATH)

    if collection_complete:
        print(RAW_PATH)

TEST MODE: no files will be saved.
